In [ ]:
#If running from start, need to download all files via webscraping; uncomment this, replace
#relevant paths, and run to download files to your 'Downloads' directory


# Need to scrape to download all files
# from selenium import webdriver

# DRIVER_PATH = '/Users/benjaminboughton/Desktop/chromedriver'
# driver = webdriver.Chrome(executable_path=DRIVER_PATH)
# driver.get('https://www.fhwa.dot.gov/policyinformation/travel_monitoring/tvt.cfm')

# '/html/body/div[6]/div[2]/div[13]/table/tbody/tr[1]/td[2]/a'
# for div in range(13,22):
#     for tr in range(1, 13):
#         print(f"\nCURRENT DIV: {div}\n")
#         print(f"\nCURRENT TR: {tr}\n")

#         driver.find_element_by_xpath(f'/html/body/div[6]/div[2]/div[{div}]/table/tbody/tr[{tr}]/td[2]/a').click()




In [136]:
import pandas as pd
import numpy as np
from os import listdir
from os.path import isfile, join


In [454]:
# Pre-define region lookup based on excel files

region_lkp = {
    'Northeast': ['Connecticut',		
                  'Maine',		
                  'Massachusetts',	
                  'New Hampshire',		
                  'New Jersey',		
                  'New York',		
                  'Pennsylvania',		
                  'Rhode Island',		
                  'Vermont'],
    'South Atlantic': ['Delaware',		
                       'District of Columbia',	
                       'Florida',		
                       'Georgia',		
                       'Maryland',		
                       'North Carolina',		
                       'South Carolina',		
                       'Virginia',		
                       'West Virginia'],
    'North Central': ['Illinois',	
                      'Indiana',		
                      'Iowa',		
                      'Kansas',		
                      'Michigan',		
                      'Minnesota',		
                      'Missouri',		
                      'Nebraska',		
                      'North Dakota',		
                      'Ohio',		
                      'South Dakota',		
                      'Wisconsin'],

    'South Gulf': ['Alabama',		
                   'Arkansas',		
                   'Kentucky',		
                   'Louisiana',		
                   'Mississippi',		
                   'Oklahoma',		
                   'Tennessee',		
                   'Texas'],

    
              
    'West': ['Alaska',		
             'Arizona',		
             'California',		
             'Colorado',		
             'Hawaii',		
             'Idaho',		
             'Montana',		
             'Nevada',		
             'New Mexico',		
             'Oregon',		
             'Utah',		
             'Washington',		
             'Wyoming'] 
}

states = [states_lst for states_lst in region_lkp.values()]

all_states = [item for sublist in states for item in sublist]




months_lkp = {'jan': 'January',
              'feb': 'February',
              'mar': 'March',
              'apr': 'April',
              'may': 'May',
              'jun': 'June',
              'jul': 'July',
              'aug': 'August',
              'sep': 'September',
              'oct': 'October',
              'nov': 'November',
              'dec': 'December'
}

years_lkp = {'02': '2002',
             '03': '2003',
             '04': '2004',
             '05': '2005',
             '06': '2006',
             '07': '2007',
             '08': '2008',
             '09': '2009',
             '10': '2010',
             '11': '2011',
             '12': '2012',
             '13': '2013',
             '14': '2014',
             '15': '2015',
             '16': '2016',
             '17': '2017',
             '18': '2018',
             '19': '2019',
             '20': '2020',
             '21': '2021'
}













In [182]:
# Get all excel files in a list
traffic_files_directory = '../../Traffic_xlsx'
xlsx_lst = [f for f in listdir(traffic_files_directory) if isfile(join(traffic_files_directory, f))]



In [455]:
# Parsing functions
def old_format_parse(original_xlsx, df_cleaned, current_month, current_year, pagename):
    df_table3 = pd.read_excel(original_xlsx, sheet_name=pagename, skiprows=1)
    column_rename_dict = {column_orig:column_orig.strip() for column_orig in list(df_table3.columns)}
    df_table3 = df_table3.rename(columns = column_rename_dict)

    

    # Fancy renaming

    if df_table3.iloc[0,3] == 'Vehicle-Miles':
        df_table3 = df_table3.rename(columns = {'Unnamed: 1': 'Region And State',
                                        'Unnamed: 2': 'Num_Stations',
                                        df_table3.columns[3]: 'Vehicle_Miles_Millions'})
    else:
        df_table3 = df_table3.rename(columns = {'Unnamed: 2': 'Region And State',
                                        'Unnamed: 3': 'Num_Stations',
                                        df_table3.columns[4]: 'Vehicle_Miles_Millions'})
    


    for state in all_states:
    
        current_state = state

        
        # Subset dataframe
        df_subset = df_table3[df_table3['Region And State'] == current_state]

        if len(df_subset) == 0:
            df_subset = df_table3[df_table3['Region And State'] == 'Dist Of Columbia']


        # Get Vehicle Miles (Millions)
        vehicle_miles = df_subset['Vehicle_Miles_Millions'].reset_index(drop=True)[0]
        # Get number of stations
        num_stations = df_subset['Num_Stations'].reset_index(drop=True)[0]
        # Get corresponding region
        current_region = [region for region in region_lkp.keys() if current_state in region_lkp[region]][0]
        # Append to cleaned_df
        current_df = pd.DataFrame([[current_state, current_region, current_year, current_month, num_stations, vehicle_miles]], columns=["State", "Region", "Year", "Month", "Num_Stations", "Vehicle_Miles_Millions"])
        df_cleaned = df_cleaned.append(current_df, ignore_index=True)

    return df_cleaned
        
def new_format_parse(original_xlsx, df_cleaned, current_month, current_year, pagename):

    
    df_page6 = pd.read_excel(original_xlsx, sheet_name=pagename, skiprows=0)

    if df_page6.iloc[1,1] == 'Number of Stations':

        df_page6 = df_page6.rename(columns = {df_page6.columns[0]: 'Region And State',
                                      df_page6.columns[1]: 'Num_Stations',
                                      df_page6.columns[2]: 'Vehicle_Miles_Millions'})
    else:
        df_page6 = df_page6.rename(columns = {df_page6.columns[0]: 'Region And State',
                                      df_page6.columns[3]: 'Num_Stations',
                                      df_page6.columns[4]: 'Vehicle_Miles_Millions'})

    for state in all_states:
        current_state = state

        
        # Subset dataframe
        df_subset = df_page6[df_page6['Region And State'] == current_state]
        
        # Get Vehicle Miles (Millions)
        vehicle_miles = df_subset['Vehicle_Miles_Millions'].reset_index(drop=True)[0]

        # Get number of stations
        num_stations = df_subset['Num_Stations'].reset_index(drop=True)[0]

        # Get corresponding region
        current_region = [region for region in region_lkp.keys() if current_state in region_lkp[region]][0]

        # Append to cleaned_df
        current_df = pd.DataFrame([[current_state, current_region, current_year, current_month, num_stations, vehicle_miles]], columns=["State", "Region", "Year", "Month", "Num_Stations", "Vehicle_Miles_Millions"])
        df_cleaned = df_cleaned.append(current_df, ignore_index=True)

    return df_cleaned




In [456]:
# Multiple xlsx
df_cleaned = pd.DataFrame(columns=["State", "Region", "Year", "Month", "Num_Stations", "Vehicle_Miles_Millions"])
for excel_file in xlsx_lst:
    print(f"current excel file: {excel_file}\n")
    original_xlsx = pd.ExcelFile(f"{traffic_files_directory}/{excel_file}")
    
    current_month = months_lkp[[month for month in list(months_lkp.keys()) if month in excel_file.lower()][0]]

    current_year = years_lkp[[year for year in list(years_lkp.keys()) if year in excel_file.lower()][0]]



    page6 = [name for name in original_xlsx.sheet_names if name.strip() == 'Page 6']
    table3 = [name for name in original_xlsx.sheet_names if name.strip() == 'Table 3']
    if page6:
        pagename = page6[0]
    elif table3:
        pagename = table3[0]



    if table3 and not page6:
        df_cleaned = old_format_parse(original_xlsx, df_cleaned, current_month, current_year, pagename)
    else:
        df_cleaned = new_format_parse(original_xlsx,df_cleaned, current_month, current_year, pagename)

    



current excel file: tvtjan03.xls

current excel file: 15novtvt.xls

current excel file: tvtjun02.xls

current excel file: 19martvt.xls

current excel file: 19jultvt.xls

current excel file: 19jantvt.xls

current excel file: 04febtvt.xls

current excel file: 12juntvt.xls

current excel file: tvtjun03.xls

current excel file: 04aprtvt.xls

current excel file: 17augtvt.xls

current excel file: 03augtvt.xls

current excel file: 10aprtvt.xls

current excel file: 06juntvt.xls

current excel file: 10febtvt.xls

current excel file: 16dectvt.xls

current excel file: tvtjan02.xls

current excel file: 20novtvt.xls

current excel file: 16octtvt.xls

current excel file: 14martvt.xls

current excel file: 18novtvt.xls

current excel file: 14jantvt.xls

current excel file: 14jultvt.xls

current excel file: tvtjul02.xls

current excel file: 03septvt.xls

current excel file: 09aprtvt.xls

current excel file: 09febtvt.xls

current excel file: 21jantvt.xls

current excel file: 17septvt.xls

current excel 

In [458]:
# df_cleaned.to_csv('../../traffic_df_all.csv')

In [177]:
# Need to scrape to download all files
# from selenium import webdriver

# DRIVER_PATH = '/Users/benjaminboughton/Desktop/chromedriver'
# driver = webdriver.Chrome(executable_path=DRIVER_PATH)
# driver.get('https://www.fhwa.dot.gov/policyinformation/travel_monitoring/tvt.cfm')

# '/html/body/div[6]/div[2]/div[13]/table/tbody/tr[1]/td[2]/a'
# for div in range(13,22):
#     for tr in range(1, 13):
#         print(f"\nCURRENT DIV: {div}\n")
#         print(f"\nCURRENT TR: {tr}\n")

#         driver.find_element_by_xpath(f'/html/body/div[6]/div[2]/div[{div}]/table/tbody/tr[{tr}]/td[2]/a').click()

# '/html/body/div[6]/div[2]/div[2]/table/tbody/tr[3]/td[2]/a'

